In [51]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound


Step 1a - Indexing (Document Ingestion)

In [52]:
video_id = "J5_-l7WIO_w"
try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript = "".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("Captions are disabled for this video.")

except NoTranscriptFound:
    print("No English transcript found for this video. Try checking if another language is available.")

No English transcript found for this video. Try checking if another language is available.


Step 1b - Indexing (Text Splitting)


In [53]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [54]:
len(chunks)

5

Step 1c & 1d - Indexing(Embedding Generation and storing in vector store)

In [55]:
embeddings = OllamaEmbeddings(
    model="all-minilm:latest"
)

vectore_store = FAISS.from_documents(chunks, embeddings)

In [56]:
vectore_store.index_to_docstore_id

{0: 'ec10264d-6c0e-43b4-a89a-9327a77130bd',
 1: 'a917547e-f950-4629-a9e0-2dc15954b446',
 2: '3455e098-ab52-4578-91ef-6ff524735d4d',
 3: 'ca6d6c1b-7355-42f9-ac5e-8dbe9b4c2f83',
 4: '54361c88-cb8b-4c20-8ca9-d9500e0cc586'}

In [57]:
vectore_store.get_by_ids(['ddb700e8-bcae-4948-8882-155511888dad'])

[]

Step 2 - Retrieval 

In [58]:
retriever = vectore_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 4}  # "k": 3 means retrieve the top 3 most relevant chunks
)

In [59]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020AFEFD8F20>, search_kwargs={'k': 4})

In [60]:
retriever.invoke("Who will teach all AI")

[Document(id='ca6d6c1b-7355-42f9-ac5e-8dbe9b4c2f83', metadata={}, page_content="and on Udemy course as well.Combinely, we serve around over 2million users on our YouTube channelsand we know how it is to teach the tech.In this course, we are going to buildsome really exciting projects. Whatabout building your own cursor CLI? Ormaybe what about building your ownknowledge graph? Or maybe what aboutbuilding a conversational voice enabledAI which talks to you, which codes foryou and does things for you. So theseall exciting projects are covered inthis course and trust me these arereally easy to build with AI agentic andagentic workflows. And at the end we arealso going to cover MCP the modelcontext protocol which was a missingpiece in the AI world. So we are goingto see what is MCP and how this enhancesthe model context and the tool callingfor these AI agents. Now this is notall. This is a constantly updated coursejust like all of our other courses andyes we will be keep on adding moreconte

Step 3. Augmentation

In [61]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}
Question: {question}
""",
    input_variables=['context', 'question']
)

In [84]:
question = "is the topic of Rag is discussed in this video? if yes then what was discussed"    
retrieved_docs = retriever.invoke(question)

In [85]:
retrieved_docs

[Document(id='54361c88-cb8b-4c20-8ca9-d9500e0cc586', metadata={}, page_content="is notall. This is a constantly updated coursejust like all of our other courses andyes we will be keep on adding morecontent into this one so that you alwaysstay updated in this era of AI. I'msuper excited to have you inside thiscourse>> and I am also super excited to see youin the course>> and together we're going to make yourjourney of learning AI super easy, supercomfortable and of course superknowledgeable.>> So see you in the course.>> Yes, of course. See you inside thecourse."),
 Document(id='ca6d6c1b-7355-42f9-ac5e-8dbe9b4c2f83', metadata={}, page_content="and on Udemy course as well.Combinely, we serve around over 2million users on our YouTube channelsand we know how it is to teach the tech.In this course, we are going to buildsome really exciting projects. Whatabout building your own cursor CLI? Ormaybe what about building your ownknowledge graph? Or maybe what aboutbuilding a conversational voice

In [86]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [87]:
final_prompt = prompt.invoke({"context":context_text, "question" : question})

Step 4 - Generation

In [88]:
llm = ChatOllama(model = "gemma4:31b-cloud", temperature=0.2)

In [90]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, RAG is discussed. The course focuses on applied AI, teaching how to use technologies like RAG and LLM models to build products that businesses can directly use.


Builiding a Chain